# Chapter 55 — PDF Processing: Extracting Useful Text from PDFs

Build a small PDF ingestion pipeline for up to five pages when we do not know the document layout in advance.
We will not assume the PDF contains a case summary, fixed headings, known table columns, or specific keywords. The pipeline will:
1. Accept a PDF path as input.
2. Inspect metadata and page count.
3. Extract selectable text page by page.
4. measure extraction quality without assuming what the content should say.
5. Detect pages that may be image-only or scanned.
6. Optionally render low-text pages and attempt OCR.
7. Save text with page references plus a diagnostic report.

# 1. Install libraries

In [15]:
%pip install -q pypdf pandas pymupdf pillow pytesseract


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# 2. Imports and output folder

In [16]:
from pathlib import Path
import json
import re
import hashlib
import pandas as pd
from pypdf import PdfReader

In [25]:
PDF_PATH = Path("Chapter 55 : Case_Summary.pdf")  # Change this to your PDF path
OUTPUT_DIR = Path("pdf_processing_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [26]:
print("Input path:", PDF_PATH.resolve())
print("Does input exist?", PDF_PATH.exists())

Input path: /Users/vimal/Documents/GitHub/Agentic_AI/Chapter 55 : Case_Summary.pdf
Does input exist? True


# 3. Validate the file and inspect metadata

We do not read the whole PDF into one giant string at the start. First we validate the path, compute a hash to identify the exact file version, and inspect the document structure.

In [28]:
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH.resolve()}\\n"
        "Update PDF_PATH to point to an existing PDF file."
    )

if PDF_PATH.suffix.lower() != ".pdf":
    raise ValueError("The input file must have a .pdf extension.")


In [32]:
file_hash = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
reader = PdfReader(str(PDF_PATH))




Core Uses of a Hash


Use Case	Description	Real-World Example
Integrity & Tamper Verification	Verifying that a file has not been altered, corrupted, or corrupted during a download. If a single bit changes, the entire hash changes completely.	When downloading software (like Linux ISOs), developers provide a checksum (hash) so you can verify the file downloaded perfectly.
Deduplication & Caching	Identifying duplicate files without comparing their entire contents line-by-line, which saves processing power and storage.	A cloud storage service (like Google Drive) checks if a file's hash already exists on their servers to avoid saving the exact same file twice.
Fast Data Lookups	Mapping complex keys to specific memory slots to find data instantly, regardless of how large the dataset grows.	Python's internal dictionary (dict) uses hashing under the hood to fetch values instantly instead of searching through a list sequentially.
Secure Data Storage	Masking sensitive information so it cannot be read in plain text. Because cryptographic hashes are one-way functions, you cannot "reverse" a hash back into the original data.	Databases store a hash of your password rather than the password itself. When you log in, it hashes your input and compares it to the stored hash.
Blockchain & Crypto	Linking blocks of data together securely. Each block contains the hash of the previous block, creating an immutable chain.	Bitcoin uses SHA-256 hashing to secure transactions and mine new blocks.


In [34]:
file_hash

'16b4413cee36cff56f0c6194ce069486f84abfbb0aad7db4e6b5913d671d5673'

In [ ]:
readerfile_hash

In [35]:
metadata = {
    "file_name": PDF_PATH.name,
    "file_size_bytes": PDF_PATH.stat().st_size,
    "sha256": file_hash,
    "page_count": len(reader.pages),
    "metadata": {str(k): str(v) for k, v in (reader.metadata or {}).items()}
}

In [36]:
metadata

{'file_name': 'Chapter 55 : Case_Summary.pdf',
 'file_size_bytes': 202275,
 'sha256': '16b4413cee36cff56f0c6194ce069486f84abfbb0aad7db4e6b5913d671d5673',
 'page_count': 5,
 'metadata': {'/Author': '(anonymous)',
  '/CreationDate': "D:20261019024030+00'00'",
  '/Creator': '(unspecified)',
  '/Keywords': '',
  '/ModDate': "D:20261019024030+00'00'",
  '/Producer': 'ReportLab PDF Library - (opensource)',
  '/Subject': '(unspecified)',
  '/Title': '(anonymous)',
  '/Trapped': '/False'}}

In [40]:
MAX_PAGES = 5
if len(reader.pages) > MAX_PAGES: 
    raise ValueError(f"This exercise is limited to {MAX_PAGES} pages, but the PDF has "
        f"{len(reader.pages)} pages. Use a five-page sample or deliberately "
        "change MAX_PAGES for a larger experiment."
    )

# 4. Extract text page by page — without assuming the content

The extractor does not search for specific words like Customer ID or Amount. It extracts whatever text the PDF exposes and records simple diagnostics for every page.

In [48]:
page_records = []

for page_number, page in enumerate(reader.pages, start = 1):
    extracted_text = page.extract_text() or ""
    cleaned_text = extracted_text.replace("\\x00", "").strip()

    page_records.append({
        "page_number" : page_number,
        "text" : cleaned_text,
        "character_count" : len(cleaned_text),
        "word_count" : len(cleaned_text.split()),
        "line_count" : sum(1 for line in cleaned_text.splitlines() if line.strip()),
        "needs_review" : len(cleaned_text) < 80})


In [49]:
page_records

[{'page_number': 1,
  'text': 'Illustrative synthetic data — not a real customer or bank investigation\nPage 1 of 5\nTRANSACTION MONITORING\nALERT CASE SUMMARY\nA practical introduction to machine learning, alerts, cases and cross-border transaction analysis\n1. What is Machine Learning (ML)?\nMachine Learning is a branch of artificial intelligence in which a system learns patterns from data to make predictions, rank risks or\nidentify unusual behaviour. Instead of relying only on manually written rules, an ML model can learn what typical activity looks like\nfrom historical examples and flag transactions that differ from expected patterns.\nML concept\nMeaning in transaction monitoring\nFeatures\nInputs such as amount, currency, country, customer type, transaction frequency and prior behaviour.\nModel / score\nA model estimates risk or unusualness; a high score is a signal to review, not proof of wrongdoing.\nAlert\nA rule or model output that highlights activity for review by an anal

In [50]:
page_df = pd.DataFrame([
    {k: v for k, v in item.items() if k != "text"}
    for item in page_records
])
display(page_df)

,page_number,character_count,word_count,line_count,needs_review
0,1,1591,240,25,False
1,2,1820,255,49,False
2,3,1484,208,42,False
3,4,1212,166,25,False
4,5,2158,301,64,False


In [54]:
# Their is always a simple way to write a code

In [51]:
full_df = pd.DataFrame(page_records)

# Neatly drop the heavy text column for display
summary_df = full_df.drop(columns=["text"])
display(summary_df)

,page_number,character_count,word_count,line_count,needs_review
0,1,1591,240,25,False
1,2,1820,255,49,False
2,3,1484,208,42,False
3,4,1212,166,25,False
4,5,2158,301,64,False


In [55]:
for item in page_records:
    print(f"\\n{'='*20} PAGE {item['page_number']} {'='*20}")
    if item["text"]:
        print(item["text"][:2500])
        if len(item["text"]) > 2500:
            print("... [preview truncated; full text is retained]")
    else:
        print("[No selectable text extracted from this page]")

\n==================== PAGE 1 ====================
Illustrative synthetic data — not a real customer or bank investigation
Page 1 of 5
TRANSACTION MONITORING
ALERT CASE SUMMARY
A practical introduction to machine learning, alerts, cases and cross-border transaction analysis
1. What is Machine Learning (ML)?
Machine Learning is a branch of artificial intelligence in which a system learns patterns from data to make predictions, rank risks or
identify unusual behaviour. Instead of relying only on manually written rules, an ML model can learn what typical activity looks like
from historical examples and flag transactions that differ from expected patterns.
ML concept
Meaning in transaction monitoring
Features
Inputs such as amount, currency, country, customer type, transaction frequency and prior behaviour.
Model / score
A model estimates risk or unusualness; a high score is a signal to review, not proof of wrongdoing.
Alert
A rule or model output that highlights activity for review by an 

# 5. Diagnose pages without knowing their layout

We can detect pages with little extracted text, but cannot conclude from that alone whether the page is blank, scanned, image-heavy, or encoded unusually. Next, use PyMuPDF to inspect image counts and render pages for possible OCR.

In [57]:
import fitz  # PyMuPDF

pdf_doc = fitz.open(str(PDF_PATH))
diagnostics = []

for page_index, page in enumerate(pdf_doc):
    text_chars = len((page.get_text("text") or "").strip())
    image_count = len(page.get_images(full=True))
    diagnostics.append({
        "page_number": page_index + 1,
        "pymupdf_text_characters": text_chars,
        "embedded_image_count": image_count,
        "low_text_flag": text_chars < 80,
        "possible_image_page": text_chars < 80 and image_count > 0
    })

diagnostics_df = pd.DataFrame(diagnostics)
display(diagnostics_df)

,page_number,pymupdf_text_characters,embedded_image_count,low_text_flag,possible_image_page
0,1,1591,0,False,False
1,2,1820,0,False,False
2,3,1484,1,False,False
3,4,1212,2,False,False
4,5,2158,1,False,False


9. What this pipeline does and does not solve
It does:
- Work without knowing the PDF's headings, schema or subject.
- Process pages individually and retain page provenance.
- Flag low-text pages for review.
- Attempt OCR only on likely problem pages, if OCR is installed.
- Save structured output suitable for a later chunking step.
It does not:
- Reliably reconstruct every complex table.
- Understand the meaning of charts or diagrams.
- Guarantee that OCR is correct.
- Infer customer, transaction or other business fields from arbitrary PDFs.
- Replace human quality checks for high-stakes documents.